In [18]:
from agents import (
    Agent,
    Runner,
    GuardrailFunctionOutput,
    RunContextWrapper,
    TResponseInputItem,
    input_guardrail,
    InputGuardrailTripwireTriggered,
    output_guardrail,
    OutputGuardrailTripwireTriggered,
)

from pydantic import BaseModel
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [13]:
class Check_Cheat(BaseModel):
    detected: bool
    explanation: str


cheat_detection_agent = Agent(
    name="Cheat Detection",
    instructions="""You are a cheat detection agent. Analyze the input for potential cheating behavior.
    like asking for answers for fill in the blanks or multiple choice questions.""",
    model="gpt-4o-mini",
    output_type=Check_Cheat,
)


@input_guardrail
async def cheat_detection_guardrail(
    ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    detection = await Runner.run(cheat_detection_agent, input)

    return GuardrailFunctionOutput(
        tripwire_triggered=detection.final_output.detected,
        output_info=detection.final_output,
    )


study_helper = Agent(
    name="Study Helper",
    instructions="You are a helpful study assistant. Provide concise and accurate answers to the user's questions based on your knowledge.",
    model="gpt-4o-mini",
    input_guardrails=[cheat_detection_guardrail],
)

In [ ]:
try:
    response = await Runner.run(
        study_helper, "Give me the names of 5 best strikers of all time in football"
    )
    print("No cheating detected.")
    print(f"Response: {response.final_output}")

except InputGuardrailTripwireTriggered as e:
    print("Cheating detected!")
    print(f"Details: {e}")

No cheating detected.
Response: Here are five of the best strikers of all time in football:

1. **Pelé**
2. **Diego Maradona**
3. **Ronaldo Nazário**
4. **Gerd Müller**
5. **Thierry Henry**

These players are renowned for their incredible goal-scoring abilities and significant contributions to the sport.


In [16]:
@output_guardrail
async def CheckForbiddenWords(
    ctx: RunContextWrapper, agent: Agent, output: str
) -> GuardrailFunctionOutput:
    forbidden_words = ["damn", "shit", "heck", "hell"]
    output = output.lower()
    found = [phrases for phrases in forbidden_words if phrases in output]
    trip_triggered = bool(found)

    return GuardrailFunctionOutput(
        tripwire_triggered=trip_triggered, output_info={"forbidden_words_found": found}
    )


agent = Agent(
    name="Customer Support Agent",
    instructions="you are customer support agent, you help users with their inquiries.",
    output_guardrails=[CheckForbiddenWords],
    model="gpt-4o-mini",
)

In [24]:
try:
    response = await Runner.run(agent, "Say heck in two syllables")
    print("No forbidden words detected.")
    print(response.final_output)
except OutputGuardrailTripwireTriggered as e:
    print("Forbidden words detected!")
    print(f"Details: {e}")

Forbidden words detected!
Details: Guardrail OutputGuardrail triggered tripwire
